# Respiration Detection Pipeline — Tuning & Evaluation Notebook

Simulate the live `VernierRespRecorder` pipeline on pre-recorded `.npz` files,
compare detections against **NeuroKit2** annotations (ground truth), and
automatically tune **all weights, filter parameters, and config values** via
[Optuna](https://optuna.org/) to maximise the combined inhalation + exhalation
F1 score.

**Workflow:**
1. Load `.npz` signal file(s)  
2. Compute NeuroKit2 peak/trough annotations (baseline)  
3. Replay signal sample-by-sample through the pipeline  
4. Evaluate detections vs baseline (precision / recall / F1 / timing error)  
5. Run Optuna parameter search  
6. Visualise best result  


In [ ]:
# Install / upgrade dependencies
import subprocess, sys
for pkg in ["neurokit2", "optuna"]:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q",
         "--break-system-packages"],
        check=False,
    )
print("Dependencies ready.")


In [ ]:
import math, warnings, glob, dataclasses
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gs
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, List, Tuple, Dict

from scipy.signal import butter, sosfilt, sosfilt_zi
import neurokit2 as nk
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.35, "axes.spines.top": False,
                     "axes.spines.right": False})
print("Imports OK.")


## ⚙️  Config

In [ ]:
@dataclass
class Config:
    srate_hz: float = 10.0

    # ── Filter ──────────────────────────────────────────────────────────
    # "l"  →  one-euro (adaptive lowpass)
    # "b"  →  Butterworth bandpass            ← default / recommended
    filter_type: str = "b"

    # One-euro lowpass params  (only used when filter_type == "l")
    min_cutoff: float = 0.3       # minimum cutoff frequency (Hz)
    beta:       float = 0.007     # speed coefficient  (higher = less lag on fast moves)
    d_cutoff:   float = 1.0       # cutoff for the derivative

    # Butterworth bandpass params  (only used when filter_type == "b")
    low:   float = 0.05           # lower edge (Hz)   ~3 bpm
    high:  float = 0.70           # upper edge (Hz)   ~42 bpm
    order: int   = 2

    # ── Derivative estimation ────────────────────────────────────────────
    deriv_tightness: int = 7      # window size in samples — must be ODD, ≥ 3
                                  # candidate sits at index deriv_tightness//2

    # ── Refractory periods ───────────────────────────────────────────────
    inh_refrac_s: float = 1.5     # minimum seconds between inhalation onsets
    exh_refrac_s: float = 1.5     # minimum seconds between exhalation onsets

    # ── Calibration ──────────────────────────────────────────────────────
    initial_calibration_delay_s: float = 25.0   # collect this many seconds before first calib
    recalibration_interval_s:    float = 10.0   # rolling recalibration cadence

    # ── Inhalation scoring  (inh onset = signal TROUGH) ──────────────────
    # Score = w1i*shape + w2i*z_inh_interval + w3i*z_exh_to_inh + w4i*z_trough_amp
    # shape     = d_f - d_i  (derivative curvature, unbounded)
    # z_*       = Gaussian z-scores in [0, 1]
    w1i:        float = 0.5    # shape curvature weight
    w2i:        float = 1.0    # inh→inh interval z-score weight
    w3i:        float = 0.8    # exh→inh cross-phase z-score weight
    w4i:        float = 0.5    # trough amplitude z-score weight
    cutoff_inh: float = 0.5    # detection threshold (score must exceed this)

    # ── Exhalation scoring  (exh onset = signal PEAK) ────────────────────
    # Score = w1e*shape + w2e*z_exh_interval + w3e*z_inh_to_exh
    w1e:        float = 0.5
    w2e:        float = 1.0
    w3e:        float = 0.8
    cutoff_exh: float = 0.5

    # ── Evaluation ───────────────────────────────────────────────────────
    match_tolerance_s: float = 1.0   # seconds within which a detection counts as a hit

print("Config defined.")


## 🔧  Signal Processing

In [ ]:
# ─── One-Euro Adaptive Lowpass ──────────────────────────────────────────────
def _sf(t_e: float, cutoff: float) -> float:
    r = 2 * math.pi * cutoff * t_e
    return r / (r + 1)

def _es(a: float, x: float, x_prev: float) -> float:
    return a * x + (1 - a) * x_prev

class lowpass:
    '''One-Euro adaptive lowpass filter (Casiez et al. 2012).'''
    def __init__(self, t0, x0, dx0=0.0, min_cutoff=1.0, beta=0.0, d_cutoff=1.0):
        self.min_cutoff = float(min_cutoff)
        self.beta       = float(beta)
        self.d_cutoff   = float(d_cutoff)
        self.x_prev     = float(x0)
        self.dx_prev    = float(dx0)
        self.t_prev     = float(t0)

    def __call__(self, t: float, x: float) -> float:
        t_e = t - self.t_prev
        if t_e <= 0:
            return self.x_prev
        a_d    = _sf(t_e, self.d_cutoff)
        dx     = (x - self.x_prev) / t_e
        dx_hat = _es(a_d, dx, self.dx_prev)
        cutoff = self.min_cutoff + self.beta * abs(dx_hat)
        a      = _sf(t_e, cutoff)
        x_hat  = _es(a, x, self.x_prev)
        self.x_prev  = x_hat
        self.dx_prev = dx_hat
        self.t_prev  = t
        return x_hat

    def filter_chunk(self, times: np.ndarray, values: np.ndarray) -> np.ndarray:
        '''Apply filter to an array of (time, value) pairs.'''
        out = np.empty(len(values))
        self.x_prev = float(values[0])
        self.t_prev = float(times[0])
        out[0] = float(values[0])
        for i in range(1, len(values)):
            out[i] = self(float(times[i]), float(values[i]))
        return out


# ─── Butterworth Bandpass ────────────────────────────────────────────────────
class bandpass:
    '''Causal Butterworth bandpass filter with persistent state (for sample-by-sample use).'''
    def __init__(self, fs: float, low: float = 0.05, high: float = 0.7, order: int = 2):
        self.sos   = butter(order, [low, high], btype="bandpass", fs=fs, output="sos")
        zi         = sosfilt_zi(self.sos)
        self.state = zi * 0.0

    def __call__(self, x_new: float) -> float:
        y, self.state = sosfilt(self.sos, np.array([x_new]), zi=self.state)
        return float(y[0])

    def filter_chunk(self, x: np.ndarray) -> np.ndarray:
        '''Apply filter to a chunk (state carries over).'''
        y, self.state = sosfilt(self.sos, np.asarray(x, dtype=float), zi=self.state)
        return y

    def reset(self):
        self.state = self.state * 0.0


def make_filter(cfg: Config):
    '''Construct a fresh filter from Config.'''
    if cfg.filter_type == "b":
        return bandpass(fs=cfg.srate_hz, low=cfg.low, high=cfg.high, order=cfg.order)
    elif cfg.filter_type == "l":
        return None   # initialised on first sample
    return None

print("Signal processing classes defined.")


## 🎯  Calibration

In [ ]:
def _spread(vals: np.ndarray) -> float:
    '''Robust spread = IQR / 1.349  (~std for Gaussian).'''
    q1, q3 = np.percentile(vals, [25, 75])
    return float((q3 - q1) / 1.349)


def _cross_intervals(start_t: np.ndarray, end_t: np.ndarray) -> np.ndarray:
    '''For each start time, find the first end time that comes after it.'''
    ivs = []
    for s in start_t:
        later = end_t[end_t > s]
        if len(later):
            ivs.append(later[0] - s)
    return np.array(ivs)


def _compute_calibration(processed_stream: list, srate_hz: float):
    '''
    Run NK2 on processed_stream to get calibration statistics.
    Returns a 12-tuple or None if not enough events found.
    '''
    sig = np.asarray(processed_stream, dtype=float)
    rsp_signals, _ = nk.rsp_process(sig, sampling_rate=srate_hz)

    trough_idx = np.flatnonzero(rsp_signals["RSP_Troughs"])   # inhalation onsets
    peak_idx   = np.flatnonzero(rsp_signals["RSP_Peaks"])     # exhalation onsets

    if len(trough_idx) < 2 or len(peak_idx) < 2:
        return None

    times_inh = trough_idx / srate_hz
    times_exh = peak_idx   / srate_hz

    ivs_inh = np.diff(times_inh);  exp_inh = np.median(ivs_inh); spr_inh = _spread(ivs_inh)
    ivs_exh = np.diff(times_exh);  exp_exh = np.median(ivs_exh); spr_exh = _spread(ivs_exh)

    iv_i2e = _cross_intervals(times_inh, times_exh)
    iv_e2i = _cross_intervals(times_exh, times_inh)
    if len(iv_i2e) < 2 or len(iv_e2i) < 2:
        return None

    exp_i2e = np.median(iv_i2e); spr_i2e = _spread(iv_i2e)
    exp_e2i = np.median(iv_e2i); spr_e2i = _spread(iv_e2i)

    trough_vals    = sig[trough_idx]
    exp_trough_val = np.median(trough_vals)
    spr_trough_val = _spread(trough_vals)

    last_inh = (float(times_inh[-1]), float(sig[trough_idx[-1]]))
    last_exh = (float(times_exh[-1]), float(sig[peak_idx[-1]]))

    return (
        exp_inh, spr_inh,
        exp_exh, spr_exh,
        exp_i2e, spr_i2e,
        exp_e2i, spr_e2i,
        exp_trough_val, spr_trough_val,
        last_inh, last_exh,
    )


def initial_calibration(processed_stream: list, srate_hz: float):
    return _compute_calibration(processed_stream, srate_hz)


def rolling_calibration(processed_stream: list, srate_hz: float):
    calib = _compute_calibration(processed_stream, srate_hz)
    if calib is None:
        return None
    return calib[:10]   # drop last_inh / last_exh

print("Calibration functions defined.")


## 🤖  SimulatedRecorder

In [ ]:
class SimulatedRecorder:
    '''
    Replays a pre-recorded numpy signal sample-by-sample,
    faithfully mirroring VernierRespRecorder live logic.

    Timestamps are synthetic:  t_i = i / srate_hz  (start at 0)
    '''

    def __init__(self, signal: np.ndarray, cfg: Config = Config()):
        self.cfg    = cfg
        self.signal = np.asarray(signal, dtype=float)
        self.dt     = 1.0 / cfg.srate_hz

    # ── Public ────────────────────────────────────────────────────────────────
    def run(self, verbose: bool = False) -> dict:
        self._reset()
        for i, val in enumerate(self.signal):
            self._step(val, i * self.dt, verbose=verbose)
        return dict(
            inh_onset         = self.inh_onset,
            exh_onset         = self.exh_onset,
            samples_processed = list(self.samples_processed),
            timestamps        = list(self.timestamps),
        )

    # ── Init ──────────────────────────────────────────────────────────────────
    def _reset(self):
        self._filt              = None
        self.samples            = []
        self.timestamps         = []
        self.samples_processed  = []
        self.inh_onset          = []
        self.exh_onset          = []
        self.last_phase         = None
        self.calibration_vals   = None
        self.initial_calib_done = False
        self.last_recalib_t     = None

    # ── Filter helper ─────────────────────────────────────────────────────────
    def _filter(self, val: float, ts: float) -> float:
        cfg = self.cfg
        if self._filt is None:
            if cfg.filter_type == "l":
                self._filt = lowpass(ts, val,
                                     min_cutoff=cfg.min_cutoff,
                                     beta=cfg.beta,
                                     d_cutoff=cfg.d_cutoff)
                return val        # first sample unfiltered
            else:
                self._filt = bandpass(fs=cfg.srate_hz,
                                      low=cfg.low, high=cfg.high, order=cfg.order)
        if cfg.filter_type == "l":
            return self._filt(ts, val) if len(self.timestamps) >= 2 else val
        else:
            return self._filt(val)

    # ── Per-sample step (mirrors _worker) ─────────────────────────────────────
    def _step(self, val: float, ts: float, verbose: bool = False):
        proc = self._filter(val, ts)
        self.samples.append(val)
        self.timestamps.append(ts)
        self.samples_processed.append(proc)

        # ── Run analysis once calibrated ──────────────────────────────────────
        if self.initial_calib_done:
            self._analysis()

        # ── Initial calibration ───────────────────────────────────────────────
        if not self.initial_calib_done and ts >= self.cfg.initial_calibration_delay_s:
            calib = initial_calibration(self.samples_processed, self.cfg.srate_hz)
            if calib is not None:
                *vals, last_inh, last_exh = calib
                self.calibration_vals   = tuple(vals)
                self.inh_onset          = [last_inh]
                self.exh_onset          = [last_exh]
                self.last_phase         = "inh" if last_inh[0] > last_exh[0] else "exh"
                self.initial_calib_done = True
                self.last_recalib_t     = ts
                if verbose:
                    print(f"  ✓ Initial calib  t={ts:.1f}s  "
                          f"({len(self.inh_onset)} inh seed, {len(self.exh_onset)} exh seed)")

        # ── Rolling recalibration ─────────────────────────────────────────────
        elif (self.initial_calib_done
              and (ts - self.last_recalib_t) >= self.cfg.recalibration_interval_s):
            n      = int(self.cfg.recalibration_interval_s * self.cfg.srate_hz)
            recent = self.samples_processed[-n:]
            if len(recent) > 0:
                calib = rolling_calibration(recent, self.cfg.srate_hz)
                if calib is not None:
                    self.calibration_vals = calib
                    self.last_recalib_t   = ts
                    if verbose:
                        print(f"  ↻ Recalibration  t={ts:.1f}s")

    # ── Derivative ────────────────────────────────────────────────────────────
    def _derivative(self):
        n = self.cfg.deriv_tightness
        if len(self.samples_processed) < n:
            return None
        mp   = n // 2
        last = self.samples_processed[-n:]
        d_i  = (last[mp] - last[0])   / (self.dt * mp)
        d_f  = (last[-1] - last[mp])  / (self.dt * mp)
        return d_i, d_f

    # ── Analysis (mirrors analysis() in VernierRespRecorder) ─────────────────
    def _analysis(self):
        if (not self.samples_processed
                or len(self.samples_processed) < self.cfg.deriv_tightness
                or self.calibration_vals is None
                or not self.inh_onset
                or not self.exh_onset):
            return

        deriv = self._derivative()
        if deriv is None:
            return

        (exp_inh,    spr_inh,
         exp_exh,    spr_exh,
         exp_i2e,    spr_i2e,
         exp_e2i,    spr_e2i,
         exp_trough, spr_trough) = self.calibration_vals

        cfg = self.cfg
        mp  = cfg.deriv_tightness // 2
        last_proc = self.samples_processed[-cfg.deriv_tightness:]
        last_ts   = self.timestamps[-cfg.deriv_tightness:]
        cand_val  = last_proc[mp]
        cand_ts   = last_ts[mp]

        last_inh_ts, _ = self.inh_onset[-1]
        last_exh_ts, _ = self.exh_onset[-1]
        dt_inh = cand_ts - last_inh_ts
        dt_exh = cand_ts - last_exh_ts

        d_i, d_f = deriv

        is_inh  = d_i < 0 and d_f > 0    # concave-up cusp → trough
        is_exh  = d_i > 0 and d_f < 0    # concave-down cusp → peak
        can_inh = self.last_phase in (None, "exh")
        can_exh = self.last_phase in (None, "inh")

        # ── Inhalation scoring ────────────────────────────────────────────────
        if is_inh and can_inh and dt_inh >= cfg.inh_refrac_s:
            z_inh    = np.exp(-0.5 * ((dt_inh    - exp_inh)    / (spr_inh    + 1e-6))**2)
            z_cross  = np.exp(-0.5 * ((dt_exh    - exp_e2i)    / (spr_e2i    + 1e-6))**2)
            z_trough = np.exp(-0.5 * ((cand_val  - exp_trough) / (spr_trough + 1e-6))**2)
            shape    = d_f - d_i   # positive for trough: rising faster than it fell
            score    = (cfg.w1i * shape
                        + cfg.w2i * z_inh
                        + cfg.w3i * z_cross
                        + cfg.w4i * z_trough)
            if score > cfg.cutoff_inh:
                self.inh_onset.append((cand_ts, cand_val))
                self.last_phase = "inh"

        # ── Exhalation scoring ────────────────────────────────────────────────
        if is_exh and can_exh and dt_exh >= cfg.exh_refrac_s:
            z_exh   = np.exp(-0.5 * ((dt_exh - exp_exh) / (spr_exh + 1e-6))**2)
            z_cross = np.exp(-0.5 * ((dt_inh - exp_i2e) / (spr_i2e + 1e-6))**2)
            shape   = d_i - d_f   # positive for peak
            score   = (cfg.w1e * shape
                       + cfg.w2e * z_exh
                       + cfg.w3e * z_cross)
            if score > cfg.cutoff_exh:
                self.exh_onset.append((cand_ts, cand_val))
                self.last_phase = "exh"

print("SimulatedRecorder defined.")


## 📂  Load .npz Data Files

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURE PATHS HERE
# ─────────────────────────────────────────────────────────────────────────────
NPZ_PATHS: List[str] = [
    # Explicit paths — add as many as you like:
    # "recordings/session_001.npz",
    # "recordings/session_002.npz",
]

NPZ_DIR: str = "."        # also glob *.npz from this directory
SRATE_HZ_OVERRIDE = None  # set to e.g. 10.0 to override the rate in all files
# ─────────────────────────────────────────────────────────────────────────────

NPZ_PATHS = list(NPZ_PATHS) + sorted(glob.glob(f"{NPZ_DIR}/*.npz"))

PREFERRED_KEYS = ("data", "samples", "signal", "respiration", "rsp", "arr_0")
SRATE_KEYS     = ("srate", "srate_hz", "fs", "sampling_rate")


def load_npz(path: str):
    '''Load (signal_array, srate_hz) from an .npz file.'''
    d   = np.load(path, allow_pickle=True)
    sig = None
    for k in PREFERRED_KEYS:
        if k in d:
            sig = d[k].flatten().astype(float); break
    if sig is None:
        first = list(d.keys())[0]
        sig   = d[first].flatten().astype(float)
        print(f"  [!] Used key '{first}' from {Path(path).name}")

    sr = SRATE_HZ_OVERRIDE
    if sr is None:
        for k in SRATE_KEYS:
            if k in d:
                sr = float(d[k]); break
    if sr is None:
        sr = 10.0
        print(f"  [!] srate not found — defaulting to {sr} Hz")
    return sig, sr


signals: Dict[str, np.ndarray] = {}
srates:  Dict[str, float]      = {}

for path in NPZ_PATHS:
    p = Path(path)
    if not p.exists():
        print(f"⚠  Not found: {path}")
        continue
    sig, sr = load_npz(str(p))
    name = p.stem
    signals[name] = sig
    srates[name]  = sr
    print(f"  ✓  {name:30s}  {len(sig):6d} samples  "
          f"({len(sig)/sr:6.1f} s  @  {sr} Hz)")

if not signals:
    print("\n⚠  No .npz files loaded. Edit NPZ_PATHS or NPZ_DIR above.")
else:
    print(f"\nTotal files: {len(signals)}")


## 🧠  NeuroKit2 Baseline Annotations

In [ ]:
def get_nk_baseline(signal: np.ndarray, cfg: Config) -> dict:
    '''
    Filter the raw signal identically to the pipeline, then run NK2 rsp_process.
    Troughs = inhalation onsets, Peaks = exhalation onsets.
    '''
    sr  = cfg.srate_hz
    sig = np.asarray(signal, dtype=float)
    ts  = np.arange(len(sig), dtype=float) / sr

    # Apply same filter (fresh instance — no state bleed)
    if cfg.filter_type == "b":
        f        = bandpass(fs=sr, low=cfg.low, high=cfg.high, order=cfg.order)
        filtered = f.filter_chunk(sig)
    elif cfg.filter_type == "l":
        f        = lowpass(ts[0], sig[0],
                           min_cutoff=cfg.min_cutoff, beta=cfg.beta, d_cutoff=cfg.d_cutoff)
        filtered = f.filter_chunk(ts, sig)
    else:
        filtered = sig.copy()

    rsp_sig, info = nk.rsp_process(filtered, sampling_rate=sr)

    peak_idx   = np.flatnonzero(rsp_sig["RSP_Peaks"])    # exh onset
    trough_idx = np.flatnonzero(rsp_sig["RSP_Troughs"])  # inh onset

    return dict(
        raw          = sig,
        filtered     = filtered,
        rsp_sig      = rsp_sig,
        peak_idx     = peak_idx,
        trough_idx   = trough_idx,
        peak_times   = peak_idx   / sr,
        trough_times = trough_idx / sr,
    )


cfg_default = Config()
baselines: Dict[str, dict] = {}

for name, sig in signals.items():
    cfg_default.srate_hz = srates[name]
    bl = get_nk_baseline(sig, cfg_default)
    baselines[name] = bl
    print(f"  {name:30s}  peaks(exh)={len(bl['peak_times']):3d}  "
          f"troughs(inh)={len(bl['trough_times']):3d}")


## 📊  Metrics & Evaluation

In [ ]:
def compute_metrics(detected_t: np.ndarray, gt_t: np.ndarray, tol_s: float = 1.0) -> dict:
    '''
    Greedy matching (nearest unmatched GT within tolerance).
    Returns precision, recall, F1, mean absolute timing error.
    '''
    det = np.sort(np.asarray(detected_t, dtype=float))
    gt  = np.sort(np.asarray(gt_t,       dtype=float))

    if len(det) == 0 or len(gt) == 0:
        return dict(precision=0., recall=0., f1=0., mae_s=np.nan,
                    tp=0, fp=len(det), fn=len(gt))

    matched_gt, matched_det, errors = set(), set(), []
    for i, d in enumerate(det):
        diffs = np.abs(gt - d)
        j     = int(np.argmin(diffs))
        if diffs[j] <= tol_s and j not in matched_gt:
            matched_gt.add(j)
            matched_det.add(i)
            errors.append(float(diffs[j]))

    tp = len(matched_det)
    fp = len(det) - tp
    fn = len(gt)  - tp
    P  = tp / (tp + fp) if tp + fp else 0.
    R  = tp / (tp + fn) if tp + fn else 0.
    F1 = 2 * P * R / (P + R) if P + R else 0.
    return dict(precision=P, recall=R, f1=F1,
                mae_s=float(np.mean(errors)) if errors else np.nan,
                tp=tp, fp=fp, fn=fn)


def evaluate(output: dict, baseline: dict, tol_s: float = 1.0) -> dict:
    '''Compare simulator output to NK2 baseline (skip seed events [0]).'''
    inh_t = np.array([t for t, _ in output["inh_onset"][1:]])
    exh_t = np.array([t for t, _ in output["exh_onset"][1:]])

    inh_m = compute_metrics(inh_t, baseline["trough_times"], tol_s)
    exh_m = compute_metrics(exh_t, baseline["peak_times"],   tol_s)

    return dict(inh=inh_m, exh=exh_m,
                combined_f1 = 0.5 * inh_m["f1"] + 0.5 * exh_m["f1"])


def run_and_evaluate(cfg: Config, signals: dict, baselines: dict,
                     verbose: bool = False) -> Tuple[dict, float]:
    '''Run on every file; return (per-file results dict, mean combined F1).'''
    results: dict = {}
    f1s: List[float] = []
    for name, sig in signals.items():
        cfg.srate_hz = srates[name]
        rec     = SimulatedRecorder(sig, cfg)
        output  = rec.run(verbose=verbose)
        metrics = evaluate(output, baselines[name], cfg.match_tolerance_s)
        results[name] = dict(output=output, metrics=metrics)
        f1s.append(metrics["combined_f1"])
        if verbose:
            m = metrics
            print(f"  {name}: F1={m['combined_f1']:.3f}  "
                  f"| inh P={m['inh']['precision']:.2f} R={m['inh']['recall']:.2f} "
                  f"mae={m['inh']['mae_s']:.2f}s "
                  f"| exh P={m['exh']['precision']:.2f} R={m['exh']['recall']:.2f} "
                  f"mae={m['exh']['mae_s']:.2f}s")
    return results, float(np.mean(f1s)) if f1s else 0.

print("Metrics functions defined.")


## 📈  Visualisation

In [ ]:
def plot_comparison(name: str, sig: np.ndarray, baseline: dict,
                    output: dict, cfg: Config, title: str = ""):
    sr  = cfg.srate_hz
    t   = np.arange(len(sig)) / sr
    sp  = np.asarray(output["samples_processed"])

    inh_t = np.array([x for x, _ in output["inh_onset"][1:]])
    inh_v = np.array([v for _, v in output["inh_onset"][1:]])
    exh_t = np.array([x for x, _ in output["exh_onset"][1:]])
    exh_v = np.array([v for _, v in output["exh_onset"][1:]])

    fig = plt.figure(figsize=(17, 10))
    gspec = gs.GridSpec(3, 1, hspace=0.45)
    fig.suptitle(f"{name}  |  {title}", fontsize=11, fontweight="bold")

    # ── Row 0: raw vs pipeline filtered ──────────────────────────────────────
    ax0 = fig.add_subplot(gspec[0])
    ax0.plot(t, sig, color="#5B9BD5", alpha=0.45, lw=0.9, label="Raw")
    ax0.plot(t, sp,  color="#1F3864", lw=1.3,          label="Pipeline filtered")
    ax0.set_ylabel("Signal")
    ax0.legend(fontsize=7, loc="upper right")
    ax0.set_title("Raw  vs  Pipeline-filtered Signal", fontsize=9)

    # ── Row 1: NK2 ground truth ───────────────────────────────────────────────
    ax1 = fig.add_subplot(gspec[1], sharex=ax0)
    ax1.plot(t, baseline["filtered"], color="#404040", lw=1)
    ax1.scatter(baseline["trough_times"],
                baseline["filtered"][baseline["trough_idx"]],
                color="#2E75B6", s=60, zorder=5, label="NK2 troughs (inh)")
    ax1.scatter(baseline["peak_times"],
                baseline["filtered"][baseline["peak_idx"]],
                color="#C00000", s=60, zorder=5, label="NK2 peaks (exh)")
    ax1.set_ylabel("Signal")
    ax1.legend(fontsize=7, loc="upper right")
    ax1.set_title("NeuroKit2 Ground Truth", fontsize=9)

    # ── Row 2: Detected vs GT overlay ────────────────────────────────────────
    ax2 = fig.add_subplot(gspec[2], sharex=ax0)
    ax2.plot(t, sp, color="#808080", lw=0.9, alpha=0.6, label="Pipeline filtered")
    # GT (faint circles)
    ax2.scatter(baseline["trough_times"],
                baseline["filtered"][baseline["trough_idx"]],
                color="#2E75B6", s=22, alpha=0.35, label="NK2 troughs")
    ax2.scatter(baseline["peak_times"],
                baseline["filtered"][baseline["peak_idx"]],
                color="#C00000", s=22, alpha=0.35, label="NK2 peaks")
    # Detected (bold)
    if len(inh_t):
        ax2.scatter(inh_t, inh_v, color="#2E75B6", s=90, marker="^",
                    zorder=6, label="Detected inh")
    if len(exh_t):
        ax2.scatter(exh_t, exh_v, color="#C00000", s=90, marker="v",
                    zorder=6, label="Detected exh")
    ax2.set_ylabel("Signal")
    ax2.set_xlabel("Time (s)")
    ax2.legend(fontsize=7, loc="upper right")
    ax2.set_title("Detected (bold ▲▼)  vs  NK2 Ground Truth (faint ●)", fontsize=9)

    plt.show()


def print_metrics_table(results: dict):
    print(f"{'File':<30} {'F1':>6} {'inh-P':>7} {'inh-R':>7} "
          f"{'inh-MAE':>8} {'exh-P':>7} {'exh-R':>7} {'exh-MAE':>8}")
    print("-" * 85)
    for name, r in results.items():
        m = r["metrics"]
        i, e = m["inh"], m["exh"]
        print(f"{name:<30} {m['combined_f1']:6.3f} "
              f"{i['precision']:7.3f} {i['recall']:7.3f} "
              f"{i['mae_s']:8.3f} "
              f"{e['precision']:7.3f} {e['recall']:7.3f} "
              f"{e['mae_s']:8.3f}")

print("Visualisation functions defined.")


## 🏃  Single Run — Default Config

In [ ]:
cfg_demo = Config()
print("Running with default config…\n")
results_demo, f1_demo = run_and_evaluate(cfg_demo, signals, baselines, verbose=True)
print(f"\nMean combined F1: {f1_demo:.4f}")
print_metrics_table(results_demo)

for name, r in results_demo.items():
    m = r["metrics"]
    plot_comparison(
        name, signals[name], baselines[name], r["output"], cfg_demo,
        title=f"[default cfg]  F1={m['combined_f1']:.3f}"
    )


## ⚙️  Parameter Tuning — Optuna

The search space covers:
- **Filter**: type, all filter-specific parameters
- **Derivative**: `deriv_tightness` (forced odd)
- **Refractory periods**: `inh_refrac_s`, `exh_refrac_s`
- **Calibration timing**: delay and interval
- **Inhalation weights**: `w1i` – `w4i`, `cutoff_inh`
- **Exhalation weights**: `w1e` – `w3e`, `cutoff_exh`

**Objective**: maximise mean combined F1 across all loaded files.  
Note: NK2 baseline is recomputed per trial so filter changes are reflected in the ground truth too.


In [ ]:
def make_cfg_from_trial(trial) -> Config:
    cfg = Config()

    # ── Filter ────────────────────────────────────────────────────────────────
    cfg.filter_type = trial.suggest_categorical("filter_type", ["l", "b"])
    if cfg.filter_type == "l":
        cfg.min_cutoff = trial.suggest_float("min_cutoff", 0.05, 2.0)
        cfg.beta       = trial.suggest_float("beta",       0.001, 0.3,  log=True)
        cfg.d_cutoff   = trial.suggest_float("d_cutoff",   0.5,   5.0)
    else:   # "b"
        cfg.low   = trial.suggest_float("bp_low",   0.02, 0.15)
        cfg.high  = trial.suggest_float("bp_high",  0.30, 1.20)
        cfg.order = trial.suggest_int  ("bp_order", 1,    5   )

    # ── Derivative  (enforce odd window) ─────────────────────────────────────
    dt_raw           = trial.suggest_int("deriv_half", 1, 7)
    cfg.deriv_tightness = 2 * dt_raw + 1    # 3, 5, 7, 9, 11, 13, 15

    # ── Refractory ────────────────────────────────────────────────────────────
    cfg.inh_refrac_s = trial.suggest_float("inh_refrac_s", 0.3, 4.0)
    cfg.exh_refrac_s = trial.suggest_float("exh_refrac_s", 0.3, 4.0)

    # ── Calibration ───────────────────────────────────────────────────────────
    cfg.initial_calibration_delay_s = trial.suggest_float("calib_delay_s", 10.0, 40.0)
    cfg.recalibration_interval_s    = trial.suggest_float("recalib_int_s", 5.0,  20.0)

    # ── Inhalation weights ────────────────────────────────────────────────────
    # NOTE: shape score units differ from z-scores (both positive when event correct)
    #       Optuna will rescale w1i relative to w2i-w4i automatically.
    cfg.w1i        = trial.suggest_float("w1i",        0.0, 5.0)
    cfg.w2i        = trial.suggest_float("w2i",        0.0, 5.0)
    cfg.w3i        = trial.suggest_float("w3i",        0.0, 5.0)
    cfg.w4i        = trial.suggest_float("w4i",        0.0, 5.0)
    cfg.cutoff_inh = trial.suggest_float("cutoff_inh", 0.0, 8.0)

    # ── Exhalation weights ────────────────────────────────────────────────────
    cfg.w1e        = trial.suggest_float("w1e",        0.0, 5.0)
    cfg.w2e        = trial.suggest_float("w2e",        0.0, 5.0)
    cfg.w3e        = trial.suggest_float("w3e",        0.0, 5.0)
    cfg.cutoff_exh = trial.suggest_float("cutoff_exh", 0.0, 8.0)

    return cfg


def objective(trial) -> float:
    if not signals:
        raise optuna.exceptions.TrialPruned()
    try:
        cfg = make_cfg_from_trial(trial)
        # Recompute NK2 baselines with the trial's filter so comparison is fair
        local_bl: dict = {}
        for name, sig in signals.items():
            cfg.srate_hz = srates[name]
            local_bl[name] = get_nk_baseline(sig, cfg)
        _, mean_f1 = run_and_evaluate(cfg, signals, local_bl, verbose=False)
        return mean_f1
    except Exception:
        raise optuna.exceptions.TrialPruned()


# ─────────────────────────────────────────────────────────────────────────────
N_TRIALS = 150    # ↑ for better results (try 300-500 for production)
# ─────────────────────────────────────────────────────────────────────────────

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=20),
    study_name="resp_detection",
)

print(f"Starting Optuna search ({N_TRIALS} trials)…")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_trial
print(f"\n✅  Best combined F1 : {best.value:.4f}")
print(f"   Trial #           : {best.number}")
print("\nBest raw params:")
for k, v in best.params.items():
    print(f"  {k:<30s}  {v}")


## 🏆  Results with Best Parameters

In [ ]:
# Build the best config
best_cfg = make_cfg_from_trial(study.best_trial)

# Recompute NK2 baselines with the best filter
best_baselines: dict = {}
for name, sig in signals.items():
    best_cfg.srate_hz = srates[name]
    best_baselines[name] = get_nk_baseline(sig, best_cfg)

# Full evaluation
best_results, best_f1 = run_and_evaluate(best_cfg, signals, best_baselines, verbose=True)
print(f"\nMean combined F1 (best cfg): {best_f1:.4f}")
print()
print_metrics_table(best_results)

for name, r in best_results.items():
    m = r["metrics"]
    title = (f"[BEST]  F1={m['combined_f1']:.3f} | "
             f"inh P={m['inh']['precision']:.2f} R={m['inh']['recall']:.2f} | "
             f"exh P={m['exh']['precision']:.2f} R={m['exh']['recall']:.2f}")
    plot_comparison(name, signals[name], best_baselines[name], r["output"],
                    best_cfg, title=title)


## 📉  Optuna Analysis

In [ ]:
# ── Optimisation history ──────────────────────────────────────────────────────
trial_vals    = [t.value for t in study.trials if t.value is not None]
best_so_far   = np.maximum.accumulate(trial_vals)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

ax = axes[0]
ax.plot(trial_vals,  alpha=0.35, color="#5B9BD5", label="Trial F1")
ax.plot(best_so_far, color="#C00000", lw=2, label="Best so far")
ax.set_xlabel("Trial"); ax.set_ylabel("Combined F1")
ax.set_title("Optimisation History"); ax.legend()

# ── Parameter importances (via fANOVA) ───────────────────────────────────────
try:
    importances = optuna.importance.get_param_importances(study)
    params      = list(importances.keys())
    imps        = list(importances.values())
    ax = axes[1]
    bars = ax.barh(params[::-1], imps[::-1], color="#5B9BD5")
    ax.set_xlabel("Importance")
    ax.set_title("Parameter Importances (fANOVA)")
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f"Importance plot unavailable: {e}")
    plt.tight_layout(); plt.show()


## 📋  Tuned Config — Copy Back to Your Codebase

In [ ]:
print("=" * 60)
print("TUNED Config  (copy these values into config.py / Config())")
print("=" * 60)
for f in dataclasses.fields(best_cfg):
    val = getattr(best_cfg, f.name)
    print(f"  {f.name:<34s} = {val!r}")
print("=" * 60)
